# Session 11 — Gaussians & Dynamics

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S11 is the **hinge into M4**. Two payoffs:

1. **The Bures–Wasserstein closed form** for Gaussians: $W_2$ between two normal
   distributions has a fully explicit formula in terms of means and covariance matrices.
2. **The Bures bridge to quantum OT**: when the means agree, the formula is *identically*
   the squared Bures distance (S7) between covariance-as-density-matrix objects.
   **Replace covariance matrices by density matrices and the same formula defines a
   quantum Wasserstein** — exactly the lift we will make in S13–S14.

Plus the **dynamic view** (Benamou–Brenier) turns $W_2$ into a Riemannian metric on
distributions — the Otto calculus that makes everything fit together.

## 0. What you'll be able to do

- Compute $W_2$ between two **multivariate Gaussians** in closed form from means and covariance matrices.
- Read the **matrix part** of the Bures–Wasserstein formula as the **Bures distance** on PSD matrices — the bridge of the course.
- Build the **affine McCann map** $T(x) = m_1 + A(x - m_0)$ that pushes one Gaussian onto another.
- Trace the $W_2$ **geodesic** in the Gaussian family — ellipses that translate, stretch, and rotate.
- Recognise the **Benamou–Brenier dynamic formulation** and Otto's Riemannian view.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ot

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import bures_distance as quantum_bures_distance
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_PLUS, KET_0, KET_1
from qot_course.transport.gaussian import (
    bures_matrix_distance, bures_wasserstein_distance,
    gaussian_geodesic, gaussian_ot_map,
)

np.random.seed(42)
viz.use_course_style()

## 1. The simplest case: 1-D Gaussians

For two normal distributions on the line, $\mathcal{N}(m_0, \sigma_0^2)$ and
$\mathcal{N}(m_1, \sigma_1^2)$, the Wasserstein-2 distance has the *elementary* closed form

$$
\boxed{\;W_2\!\left(\mathcal{N}(m_0, \sigma_0^2), \mathcal{N}(m_1, \sigma_1^2)\right)
       \,=\, \sqrt{(m_0 - m_1)^2 + (\sigma_0 - \sigma_1)^2}.\;}
$$

The two contributions are clearly separated: a **translation cost** $|\Delta m|$ and a
**rescaling cost** $|\Delta \sigma|$. The Wasserstein distance is the Euclidean distance
in the $(m, \sigma)$ plane — *the geometry on Gaussians is flat in those coordinates*.

In [ ]:
m_0, sigma_0 = 0.0, 1.0
m_1, sigma_1 = 4.0, 2.0
w2_closed = bures_wasserstein_distance([m_0], [[sigma_0**2]], [m_1], [[sigma_1**2]])
w2_formula = float(np.sqrt((m_0 - m_1) ** 2 + (sigma_0 - sigma_1) ** 2))
print(f"W_2 via Bures–Wasserstein code  = {w2_closed:.6f}")
print(f"W_2 via 1-D formula             = {w2_formula:.6f}")
print(f"agreement?                       {np.isclose(w2_closed, w2_formula)}")

# Cross-check via discretised LP.
grid = np.linspace(-5.0, 12.0, 400)
def gaussian_density_1d(x, m, s):
    return np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))
mass_0 = gaussian_density_1d(grid, m_0, sigma_0); mass_0 = mass_0 / mass_0.sum()
mass_1 = gaussian_density_1d(grid, m_1, sigma_1); mass_1 = mass_1 / mass_1.sum()
cost_grid = (grid.reshape(-1, 1) - grid.reshape(1, -1)) ** 2
w2_lp = float(np.sqrt(ot.emd2(mass_0, mass_1, cost_grid)))
print(f"W_2 via grid-discretised LP     = {w2_lp:.6f}")

**Read the output.** All three numbers agree: the closed form, the implementation, and
the brute-force LP. The Gaussian world is the *simplest* place where Wasserstein has a
fully explicit formula — and it generalises beautifully to multi-D.

## 2. Multivariate Gaussians — the Bures–Wasserstein formula

For two multivariate Gaussians $\mathcal{N}(m_0, \Sigma_0)$ and $\mathcal{N}(m_1, \Sigma_1)$
on $\mathbb{R}^d$ (Dowson & Landau, 1982; Olkin & Pukelsheim, 1982; Givens & Shortt, 1984):

$$
\boxed{\;W_2^2 \,=\, \|m_0 - m_1\|^2 \,+\, \mathrm{tr}(\Sigma_0) + \mathrm{tr}(\Sigma_1)
       - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}.\;}
$$

The first term is the **squared translation** (mean shift); the second term — the **Bures
matrix distance** — measures how the two covariance matrices differ in shape and
orientation. Let's verify the formula on a 2-D example.

In [ ]:
# Two 2-D Gaussians: different positions AND different covariance shapes.
mean_0 = np.array([-2.5, -1.0])
cov_0  = np.array([[1.5, 0.4], [0.4, 0.6]])    # elongated, tilted right
mean_1 = np.array([ 2.5,  1.5])
cov_1  = np.array([[0.5, -0.3], [-0.3, 1.4]])  # elongated, tilted left

w2 = bures_wasserstein_distance(mean_0, cov_0, mean_1, cov_1)
mean_term   = float(np.sum((mean_0 - mean_1) ** 2))
matrix_term = bures_matrix_distance(cov_0, cov_1)
print(f"||m_0 - m_1||^2                           = {mean_term:.4f}")
print(f"bures_matrix_distance(Sigma_0, Sigma_1)    = {matrix_term:.4f}")
print(f"W_2 (closed form)                          = {w2:.4f}")

# Cross-check via grid LP on a 20x20 mesh.
grid = np.linspace(-7.0, 7.0, 20)
X, Y = np.meshgrid(grid, grid)
pts = np.stack([X.ravel(), Y.ravel()], axis=-1)
def gaussian_density_2d(pts, mean, cov):
    diff = pts - mean
    inv = np.linalg.inv(cov)
    return np.exp(-0.5 * np.einsum("ij,jk,ik->i", diff, inv, diff))
p0 = gaussian_density_2d(pts, mean_0, cov_0); p0 = p0 / p0.sum()
p1 = gaussian_density_2d(pts, mean_1, cov_1); p1 = p1 / p1.sum()
cost_2d = np.sum((pts[:, None, :] - pts[None, :, :]) ** 2, axis=-1)
w2_lp = float(np.sqrt(ot.emd2(p0, p1, cost_2d)))
print(f"\nW_2 (grid-LP cross-check, 20x20 mesh)      = {w2_lp:.4f}")
print(f"agreement within grid resolution?           {np.isclose(w2, w2_lp, rtol=0.05)}")

**Read the output.** Closed form and grid LP agree within the discretisation error of
the $20 \times 20$ mesh. The **mean term** captures the cost of moving the centre of
mass; the **matrix term** captures the cost of reshaping the covariance. The big
question now: *what is that matrix term?*

## 3. The matrix term is the Bures distance — the bridge to quantum

Look at the matrix term again:

$$ \mathrm{tr}(\Sigma_0) + \mathrm{tr}(\Sigma_1) - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}. $$

It is defined for **any pair of Hermitian PSD matrices**, not just covariance matrices.
And for **unit-trace** PSD matrices — i.e. density matrices $\rho_0, \rho_1$ — it
becomes

$$ 1 + 1 - 2\,\underbrace{\mathrm{tr}\sqrt{\sqrt{\rho_0}\,\rho_1\,\sqrt{\rho_0}}}_{F_U(\rho_0, \rho_1)\ \text{(Uhlmann fidelity)}}
   = 2\,(1 - F_U(\rho_0, \rho_1)) = d_B^2(\rho_0, \rho_1). $$

That is **exactly** the squared Bures distance from S7. So the **matrix part of the
Bures–Wasserstein formula on Gaussians, restricted to unit-trace matrices, is the Bures
distance on density matrices**. Let us verify it numerically.

In [ ]:
pairs = [
    ("|+><+|  vs  I/2",  density_matrix(KET_PLUS), maximally_mixed(2)),
    ("|0><0|  vs  |+><+|", density_matrix(KET_0), density_matrix(KET_PLUS)),
    ("|0><0|  vs  |1><1|", density_matrix(KET_0), density_matrix(KET_1)),
]
print(f"{'pair':<22s} {'d_B (S7)':>12s} {'sqrt(BW)(S11)':>16s} {'match?':>8s}")
print("-" * 60)
for label, rho, sigma in pairs:
    d_b_s7 = quantum_bures_distance(rho, sigma)
    bw_matrix = bures_matrix_distance(rho, sigma)
    sqrt_bw = float(np.sqrt(max(0.0, bw_matrix)))
    print(f"{label:<22s} {d_b_s7:>12.4f} {sqrt_bw:>16.4f} {str(np.isclose(d_b_s7, sqrt_bw, atol=1e-6)):>8s}")

**Read the table.** For three different pairs of qubit density matrices — including the
classic $|+\rangle\langle+|$ vs $I/2$ "same-diagonal-different-state" case — the
S7 Bures distance **exactly** equals the square root of the S11 Bures–Wasserstein
matrix term. The two formulas are the same object, viewed from two sides:

- On **classical Gaussians**, the formula gives Wasserstein-2 distance.
- On **quantum density matrices**, the same formula gives the Bures distance.

**This is the bridge to QOT.** In S13 we will *define* quantum Wasserstein by lifting
the LP marginals from $a, b$ (mass vectors) to $\rho_A, \rho_B$ (density matrices), and
this matrix-fidelity expression will sit naturally inside the SDP. The Bures distance is
the *quantum Wasserstein on commuting states* — a special case we already know how to
compute.

## 4. The W₂ geodesic in the Gaussian family — an affine McCann map

For zero-mean Gaussians, the **optimal transport map** is affine and explicit:

$$ T(x) = A\,x, \qquad A = \Sigma_0^{-1/2}\bigl(\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}\bigr)^{1/2}\Sigma_0^{-1/2}. $$

$A$ is the unique symmetric positive-definite matrix that satisfies $A \Sigma_0 A = \Sigma_1$.
For general means, $T(x) = m_1 + A(x - m_0)$. The $W_2$ **geodesic** therefore stays in
the Gaussian family:

$$ \mu_t = \mathcal{N}\!\left((1 - t)\,m_0 + t\,m_1,\; M_t\,\Sigma_0\,M_t^\top\right),
   \qquad M_t = (1 - t) I + t A. $$

The interpolation **rotates, translates, and rescales** the covariance ellipsoid — all
at once, in one closed form. Let's draw it.

In [ ]:
def ellipse_from_gaussian(mean, cov, n_std=2.0, **kwargs):
    vals, vecs = np.linalg.eigh(0.5 * (cov + cov.T))
    order = vals.argsort()[::-1]
    vals = vals[order]; vecs = vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = (2.0 * n_std * np.sqrt(np.clip(vals, 0.0, None))).tolist()
    return patches.Ellipse(xy=tuple(mean), width=width, height=height,
                           angle=angle, **kwargs)

# OT map from cov_0 to cov_1.
A = gaussian_ot_map(cov_0, cov_1)
pushed = A @ cov_0 @ A.T
print(f"A Sigma_0 A == Sigma_1 ?  {np.allclose(pushed, cov_1, atol=1e-9)}")
print(f"A symmetric (SPD)?         {np.allclose(A, A.T)}")
print(f"A eigenvalues > 0 ?        {np.all(np.linalg.eigvalsh(A) > 0)}")
print()

# Geodesic snapshots.
ts = np.linspace(0.0, 1.0, 9)
snapshots = [gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, t) for t in ts]
means = np.array([s[0] for s in snapshots])

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.colormaps["viridis"]
for k, (t, (mean_t, cov_t)) in enumerate(zip(ts, snapshots)):
    ax.add_patch(
        ellipse_from_gaussian(
            mean_t, cov_t, n_std=1.5,
            edgecolor=cmap(k / (len(ts) - 1)),
            facecolor=cmap(k / (len(ts) - 1)),
            alpha=0.18 + 0.5 * (k / (len(ts) - 1)),
            linewidth=2.0,
            label=f"t = {t:.2f}" if k in (0, len(ts) // 2, len(ts) - 1) else None,
        )
    )
ax.plot(means[:, 0], means[:, 1], "--", color=COLORS["text"], lw=1.2, label="mean trajectory")
ax.scatter(means[[0, -1], 0], means[[0, -1], 1], color=COLORS["text"], s=70, zorder=4)
ax.set_xlim(-6, 6); ax.set_ylim(-5, 5); ax.set_aspect("equal")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("$W_2$ geodesic in the Gaussian family — translation + rotation + rescaling all at once", pad=12)
ax.legend(loc="upper left")
plt.show()

**Read the figure.** The OT map $A$ pushes $\Sigma_0$ exactly onto $\Sigma_1$ as
verified above. The **geodesic ellipses** flow smoothly from the lower-left Gaussian
(violet) to the upper-right Gaussian (yellow), the mean tracking a straight line in
$\mathbb{R}^2$ and the covariance shape blending in a way that **stays Gaussian at every
$t$**. This is a luxury we lose for general distributions — and exactly the reason
Gaussians are the playground where everything is computable. *(Same affine-map gift
shows up in S14: the optimal quantum coupling between Gaussian states will be expressible
in a closed-form analogous expression.)*

## 5. The dynamic view — Benamou–Brenier and Otto calculus

The static OT problem has a beautiful **dynamic reformulation** (Benamou & Brenier, 2000):

$$
\boxed{\;W_2^2(\mu_0, \mu_1) \,=\, \inf_{(\rho_t, v_t)}\
       \int_0^1\!\!\int |v_t(x)|^2\,\rho_t(x)\,\mathrm{d}x\,\mathrm{d}t\;}
$$

subject to the **continuity equation** $\partial_t \rho_t + \nabla\!\cdot\!(\rho_t\,v_t) = 0$,
$\rho_0 = \mu_0$, $\rho_1 = \mu_1$.

In words: find a *path of distributions* $\rho_t$ and a *velocity field* $v_t$ that move
the mass from $\mu_0$ to $\mu_1$, with the lowest total kinetic energy. The unique
optimal velocity field is the **gradient** of a Kantorovich potential (Brenier);
$\rho_t$ is the McCann interpolation.

**Otto (2001)** observed that this gives the space of distributions
$\mathcal{P}_2(\mathbb{R}^d)$ a **Riemannian-like structure**: the tangent space at
$\rho$ is identified with velocity fields, the metric is
$\langle v_1, v_2\rangle_\rho = \int \langle v_1, v_2\rangle \rho\,\mathrm{d}x$, and
geodesics are McCann interpolations. **The Wasserstein space is a curved geometry**,
just like the Fisher–Rao geometry of S6 was a curved geometry on probability
distributions — but **respecting the ground space**.

For the **quantum lift (S14)**, the continuity equation will become a **Lindblad-type
evolution** on density matrices, and Otto's metric will become a *quantum* Riemannian
metric — the bridge survives one more time.

## 6. Dictionary update — the Bures bridge is explicit

| Classical | Quantum |
|-----------|---------|
| Gaussian $\mathcal{N}(m, \Sigma)$ on $\mathbb{R}^d$ | Gaussian state on the harmonic oscillator (S14 mention) |
| **covariance matrix** $\Sigma$ (PSD, finite trace) | **density matrix** $\rho$ (PSD, unit trace) |
| **$W_2$ closed form** $\|\Delta m\|^2 + \mathrm{tr}(\Sigma_0) + \mathrm{tr}(\Sigma_1) - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\Sigma_1\Sigma_0^{1/2}}$ | **Bures distance** $d_B^2(\rho_0, \rho_1) = 2(1 - F_U) = $ same formula on unit-trace PSD |
| **McCann map** $T(x) = m_1 + A(x - m_0)$ | **quantum coupling map** *(S13)* |
| **$W_2$ geodesic** on Gaussians | **Bures geodesic** on density matrices (already implicit in S7 / Uhlmann) |
| Benamou–Brenier dynamic form | **quantum continuity equation** *(S14)* |
| Otto calculus on $\mathcal{P}_2$ | **quantum Riemannian metrics on $\mathcal{S}(\mathcal{H})$** *(S14; Carlen–Maas)* |

## 7. Your turn

1. **The 1-D Gaussian "flat plane".** Pick a grid of $(m, \sigma)$ values, compute the
   pairwise $W_2$ matrix using the 1-D closed form, and use multidimensional scaling
   (e.g. `sklearn.manifold.MDS`) to embed it. Does it lie flat in $\mathbb{R}^2$? *Hint:
   yes — that is exactly the $(m, \sigma)$ plane.*
2. **Mean-only vs covariance-only.** Build $\mathcal{N}(m_0, \Sigma_0)$ and
   $\mathcal{N}(m_0, \Sigma_1)$ (same mean, different covariance). Verify numerically that
   $W_2$ reduces to $\sqrt{\text{bures\_matrix\_distance}(\Sigma_0, \Sigma_1)}$, and that
   this equals the S7 Bures distance when both are unit-trace.
3. **Eigenvectors of $A$.** Compute the eigendecomposition of the OT map $A$ between two
   diagonal covariances $\Sigma_0 = \mathrm{diag}(1, 4)$ and
   $\Sigma_1 = \mathrm{diag}(9, 1)$. Predict the eigenvalues (hint: $\sigma_1 / \sigma_0$
   in each direction) and check.

## Conclusions & references

- **$W_2$ has a closed form on Gaussians**: a translation cost in mean + a Bures matrix
  cost on covariances.
- **The matrix cost is the Bures distance** on PSD matrices. Restricted to unit-trace
  matrices it is **identically the S7 Bures distance on density matrices**. *This is the
  bridge to QOT.*
- The **McCann map between Gaussians is affine**; the $W_2$ geodesic stays Gaussian and
  is fully explicit.
- **Benamou–Brenier / Otto** turn $W_2$ into a Riemannian metric on distributions; in
  S14 this lifts to a *quantum* Riemannian metric on density matrices.
- **M3 ends here. Movement 4 (Quantum optimal transport) begins next session.**
- **Next (S12 — Why QOT):** the diagonal-collapse lesson, the plus-state-vs-mixed example,
  Trevisan's taxonomy of quantum OT formulations, and the *finally-completed* dictionary.

**References:** D. C. Dowson, B. V. Landau, "The Fréchet distance between multivariate
normal distributions", J. Multivar. Anal. **12**, 450 (1982); I. Olkin, F. Pukelsheim,
"The distance between two random vectors with given dispersion matrices", Linear Algebra
Appl. **48**, 257 (1982); J.-D. Benamou, Y. Brenier, "A computational fluid mechanics
solution to the Monge–Kantorovich mass transfer problem", Numer. Math. **84**, 375
(2000); F. Otto, "The geometry of dissipative evolution equations: the porous medium
equation", Comm. PDE **26**, 101 (2001); R. Bhatia, T. Jain, Y. Lim, "On the Bures–Wasserstein
distance between positive definite matrices", Expo. Math. **37**, 165 (2019). Previous:
`notebooks/03_ClassicalOptimalTransport/03_sinkhorn.ipynb`.